# Using Snowpark DataFrames

### Topics in this lesson


1. [What Is a DataFrame?](#What_Is_a_DataFrame)  
1. [Different Ways to Create a DataFrame](#Ways_to_Create_a_DataFrame)    
    1. [DataFrame from Client-Side Collections](#From_Client-Side_Collections)  
        1. [Session.create_dataframe(...)](#create_dataframe)  
    1. [DataFrame from Tables and Views](#From_Tables_and_Views)  
        1. [Creating `DataFrame` Objects Using `Session.table(...)`](#session_table)
    1. [DataFrame from Staged Files](#From_Staged_Files)  
        1. [What is a `DataFrameReader`?](#DataFrameReader)   
        1. [Configuring Readers](#Configuring_Readers)   
    1. [DataFrame from Raw SQL Statements](#From_Raw_SQL_Statements)  
        1. [Creating `DataFrame` Objects Using `Session.sql(...)`](#Session_SQL)  
1. [DataFrame Schemas and Data Types](#DataFrame_Schemas_and_Data_Types)  
    1. [Snowpark Data Types and Snowflake SQL Data Types](#Snowpark_SQL_Datatypes)  
    1. [Schemas](#Schemas)  
1. [Introduction to DataFrame Transformations](#intro_dataframe_transformations)  
    1. [Accessing `Column` Objects](#Accessing_Columns)  
    1. [Using the `snowflake.snowpark.functions` Object   ](#Using_the_com_snowflake_snowpark_functions_Object)  
    1. [Adding a `Column` Using `DataFrame.with_column(...)` and `.with_columns(...)`](#Adding_a_Column_Using_DataFrame_with_column_and_withColumns)  
    1. [`Column` Mathematical Operators (`+`,`-`,`*`,`/` and `%`)](#Math_operators)  
    1. [Logical, Relational and Comparison Operators (`<`, `>`, `!=`, `&` etc.)](#Logical_Relational_and_Comparison_Operators)  
    1. [Casting and Aliasing `Column` Objects](#Column_Expression_Functions)  
        1. [Casting a `Column`](#Casting_a_Column)  
        1. [Aliasing a `Column` Name](#Aliasing_a_Column_Name)  
    1. [Aggregating Data Using `DataFrame.group_by(...).agg(...)`](#Aggregating_Data_Using_DataFrame_group_by_agg)  
1. [DataFrame Actions](#DataFrame_Actions)  
    1. [DataFrame Collect](#DataFrame_collect)  
    1. [Creating Views from DataFrames](#Creating_Views_from_DataFrames)  
    1. [Writing a `DataFrame` to a Table](#Writing_a_DataFrame_to_a_Table)  
1. [Explain And Describe](#Explain_Plans)      
    1. [Explain And Describe](#Explain_Plans)      
    1. [Using `DataFrame.describe()`](#DataFrame_describe)      
1. [Clean Up and Close Session](#L2_cleanup)      
    


### MyNote: Test data

use script created by CLINE: 4_prepare_data/Lectures/create_tpch_sf10_test_data.sql

---
<a id="What_Is_a_DataFrame"></a>

## 1. What Is a DataFrame?

<img src="../../images/AnatomyOfADataFrame.png" alt="AnatomyOfADataFrame" style="width:85%;display:block;margin-left:5%;" />

> **&#128221; Note:** For full details of Snowpark-specific `DataFrame` capabilities, see: [snowflake.snowpark.DataFrame](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.DataFrame.html#snowflake-snowpark-dataframe)

---
<a id="Ways_to_Create_a_DataFrame"></a>
## 2. Different Ways to Create a DataFrame

`DataFrame` objects can be created using a variety of create operations.
1. From Client-Side Collections
1. From Snowflake Tables and Views
1. From Staged Files
1. From Raw SQL Statements

<img src="../../images/DFOperationsCreatePython.png" alt="DFOperationsCreate" style="width:85%;display:block;margin-left:5%;" />

> **&#128221; Note:** In this lesson, the action operation `DataFrame.show()` will be invoked to evaluate `DataFrame` operations and print results to the output. Action operations, including `.show()` and others, will  be covered in more detail in a later lecture.

#### Connect and create a `Session`

The following cell connects to your Snowflake account and creates an instance of `Session`. 

*You needn't modify anything in this cell. Just run it.*
> &#10071; Success requires that you have already completed the key pair authentication exercise.

In [1]:
# Run utils notebook
%run ../../utils/ds_utils_python_MINE.ipynb

# Connect to Snowflake and create a Session object named session
# session = create_session()

In [2]:
# My code for Snowflake account connection

CONFIG_DIR = '/Users/richardkirk/.ssh'
CONFIGFILE = CONFIG_DIR + '/sf_config'


# Load configuration file
with open(CONFIGFILE) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], "rb") as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{"private_key": private_key_bytes}}).create()

---
#### Setup Code

We should set ourselves up for success. The following ensures our context is set properly for database, schema, role, and warehouse.

*You needn't edit anything in the following cell. Just run it.*

In [3]:
# Hard code the lesson name
lesson_name = "CREATING_DATAFRAMES_PY"

# Create the context items for this lesson
lesson = confirm_or_create_lesson_context(session, lesson_name)

Creation of context items could take a few moments... be patient
The current user is: RKIRK
Query tag is: Data Science: RKIRK - CREATING_DATAFRAMES_PY
-------------------------------------------------
|"Created warehouse RKIRK_WH"                   |
-------------------------------------------------
|RKIRK_WH already exists, statement succeeded.  |
-------------------------------------------------

------------------------------------
|"Altered warehouse RKIRK_WH"      |
------------------------------------
|Statement executed successfully.  |
------------------------------------

Setting current warehouse to RKIRK_WH
Creating database RKIRK_DB
-------------------------------------------------
|"Created database RKIRK_DB"                    |
-------------------------------------------------
|RKIRK_DB already exists, statement succeeded.  |
-------------------------------------------------

Creating schema CREATING_DATAFRAMES_PY_LESSON
--------------------------------------------------

In [ ]:

# MyCode - checking session context
session.get_current_warehouse()
session.get_current_database()
session.get_current_schema()
session.get_current_role()

'"ACCOUNTADMIN"'

<a id="From_Client-Side_Collections"></a>
### 2A. DataFrame from Client-Side Collections

<a id="create_dataframe"></a>

#### 2Aa. Session.create_dataframe(...)

Provide a `List` or `Tuple` of data. Optionally, you may provide a schema. The schema will be inferred from the data, and the columns will be named dynamically if no schema is provided.

The following creates `DataFrame`s from a `List` and a `Tuple` of data. The single column in each will be named `_1`.
```python
# Provide a List
my_DF = (session
    .create_dataframe(
        [value1, value2, value3] # <-- List
    )
)

# Provide a Tuple
my_DF = (session
    .create_dataframe(
        (value1, value2, value3)  # <-- Tuple
    )
)  
```

The columns can be named during creation by optionally providing a simple list of strings as the `schema` parameter.

```python
# Provide a List of data and a schema 
my_DF = (session
    .create_dataframe(
         [(value1A, value1B, valueC), (value2A, value2B, valueC)]
        ,schema = ["col1", "col2", "col3"]
    )
)
```


> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.Session.create_dataframe(List)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Session.create_dataframe.html#snowflake.snowpark.Session.create_dataframe)
> - [Python `list`](https://docs.python.org/3.8/library/stdtypes.html#list)

In [ ]:
# Create a DateFrame from a list of strings
# MyNote: nothing happens server side here
df_list = (session
    .create_dataframe(
        ["some", "data", "in", "a list", "chicken duck"]
    )
)       

# View the rows of data
df_list.show()

----------------
|"_1"          |
----------------
|some          |
|data          |
|in            |
|a list        |
|chicken duck  |
----------------



## MyNote: .show() Server side execution

`.show()` — triggers server-side execution. Snowpark generates the below SQL which can be seen in snowsight query Query History:


```sql
SELECT 
    "_1"
 FROM (
 SELECT $1 AS "_1" FROM  VALUES ('some' :: STRING), ('data' :: STRING), ('in' :: STRING), ('a list' :: STRING), ('chicken duck' :: STRING)
)
```

In [10]:
# Create a DateFrame from a list of strings
df_list = (session
    .create_dataframe(
        ["some", "data", "in", "a list"]
    )
)       

# View the rows of data
df_list.show()

# Create a DateFrame from a list of tuples and provide a schema
df_list_list_of_tuples = (session
    .create_dataframe(
         [("row1 col1", "row1 col2"), ("row2 col1", "row2 col2")]
        ,schema = ["col1", "col2"]
    )
)

# View the rows of data
df_list_list_of_tuples.show()

----------
|"_1"    |
----------
|some    |
|data    |
|in      |
|a list  |
----------

-------------------------
|"COL1"     |"COL2"     |
-------------------------
|row1 col1  |row1 col2  |
|row2 col1  |row2 col2  |
-------------------------



Additionally, provide a `StructType` for the schema along with the collection of row data.

```python
# Provide a StructType with List
from snowflake.snowpark.types import StructType, StructField, XXXType

my_schema = 
    StructType(
        [
          StructField(<name1>,XXXType())
         ,StructField(<name2>,XXXType())
         ,StructField(<name3>,XXXType())
        ]
      )
my_row_data = [...]
my_DF1 = session
    .create_dataframe(my_row_data,my_schema) # Create DataFrame from provided list and schema
```

> **&#128221; Note:** See documentation for further details:
> - [Data Types](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/types.html#data-types)
> - [snowflake.snowpark.types.StructType](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.types.StructType.html#snowflake-snowpark-types-structtype)
> - [snowflake.snowpark.types.StructField](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.types.StructField.html#snowflake-snowpark-types-structfield)

In [11]:
# We need to import some data types to make our code more manageable 
from snowflake.snowpark.types import StructType, StructField, IntegerType, StringType

# Each tuple has two values 1) integer 2) str
row_data_list = [
          ( 987, "a"), (453, "b"), ( 45, "c")
         ,( 257, "d"), (  1, "e"), (890, "f")
         ,(1424, "g"), ( 16, "h"), ( 22, "i")
         ,( 222, "j"), (-87, "k"), ( -1, "l")
]    

# Define the schema for the columns of the DataFrame.
schema = (
    StructType(
        [
             StructField("num_col", IntegerType()) # First Column
            ,StructField("text_col", StringType()) # Second Column
        ]
     )
)

# Create the DataFrame
list_and_schema_df = (session
    .create_dataframe(row_data_list, schema)
)           

# Print some rows to standard out
list_and_schema_df.show()

--------------------------
|"NUM_COL"  |"TEXT_COL"  |
--------------------------
|987        |a           |
|453        |b           |
|45         |c           |
|257        |d           |
|1          |e           |
|890        |f           |
|1424       |g           |
|16         |h           |
|22         |i           |
|222        |j           |
--------------------------



<a id="From_Tables_and_Views"></a>
### 2B DataFrame from Tables and Views

<a id="session_table"></a>

#### 2Ba. Creating `DataFrame` Objects Using `Session.table(...)`

- Use `Session.table(...)` for both Snowflake tables and views
- `RelationalGroupedDataFrame` is covered in detail in Lecture 04-Transforming-Data-II
- Returns a `Table` (child of `DataFrame` class)
    - `DataFrame` : immutable (no DML)
    - `Table` : mutable (updates, deletes, merges)

<img src="../../images/PythonDataFrameClassHierarchy.png" alt="PythonDataFrameClassHierarchy" style="width:40%;display:block;margin-left:10%;" />


```python
my_table_DF = (session
    .table("<name of table or view>")
)
```



> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.Session.table(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Session.table.html#snowflake.snowpark.Session.table)
> - [snowflake.snowpark.Table](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Table.html#snowflake.snowpark.Table)
> - [snowflake.snowpark.DataFrame](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.DataFrame.html#snowflake-snowpark-dataframe)

In [12]:
# Create a Table object from a Snowflake table using Session.table(...)
cust_table_df = (session
    .table("TRAINING_DB.TPCH_SF10.CUSTOMER")
)               

print(f"The class of custTableDF is {type(cust_table_df)}")
 
# Create a Table object from a Snowflake view using Session.table(...)
stages_view_DF = (session
    .table("TRAINING_DB.INFORMATION_SCHEMA.STAGES")
)               

print(f"The class of stages_view_DF is {type(stages_view_DF)}")

The class of custTableDF is <class 'snowflake.snowpark.table.Table'>
The class of stages_view_DF is <class 'snowflake.snowpark.table.Table'>


<a id="From_Staged_Files"></a>

### 2C. DataFrame from Staged Files

<a id="DataFrameReader"></a>

#### 2Ca. What is a `DataFrameReader`?

- Used to create `DataFrame`s from staged files
- `Session.read` function creates a `DataFrameReader`
    - Configure the reader's options based on the staged file format
    - Provide a `StructType` for the schema (for CSV file format)
    - Read from the files
- Each file format has its own [file format options](https://docs.snowflake.com/en/sql-reference/sql/create-file-format.html#format-type-options-formattypeoptions)
- Supported file formats are:
    - Avro ([options](https://docs.snowflake.com/en/sql-reference/sql/create-file-format.html#type-avro))
    - CSV ([options](https://docs.snowflake.com/en/sql-reference/sql/create-file-format.html#type-csv))
    - JSON ([options](https://docs.snowflake.com/en/sql-reference/sql/create-file-format.html#type-json))
    - ORC ([options](https://docs.snowflake.com/en/sql-reference/sql/create-file-format.html#type-orc))
    - Parquet ([options](https://docs.snowflake.com/en/sql-reference/sql/create-file-format.html#type-parquet))
    - XML  ([options](https://docs.snowflake.com/en/sql-reference/sql/create-file-format.html#type-xml)) (in [preview](https://docs.snowflake.com/en/release-notes/preview-features.html))

<a id="Configuring_Readers"></a>

#### 2Cb. Configuring Readers

- `DataFrameReader` uses a builder pattern
- Chain function calls for elegant code

```python
# Create and configure reader with method chaining
my_DFR = (
    session.read
        .schema(...)
        .option(...)
        .option(...) # Returns a DataFrameReader
)    
```

- Configure reader options based on file format
    - Provide a schema for CSV files
    - JSON files return data in a column of type `VARIANT`
    - Schema inferred from embedded data for binary formats
- Call the appropriate method for the file format
- Chain of methods returns a DataFrame

> **&#128221; Note:** See documentation for further details:
> - [CREATE FILE FORMAT - Type CSV](https://docs.snowflake.com/en/sql-reference/sql/create-file-format.html#type-csv)
> - [CREATE FILE FORMAT - Type JSON](https://docs.snowflake.com/en/sql-reference/sql/create-file-format.html#type-json) 
> - [Apache&reg; Parquet File Format](https://parquet.apache.org/)
> - [Apache&reg; Avro File Format](https://avro.apache.org/)
> - [Apache&reg; ORC File Format](https://orc.apache.org/)

### MyNote: Example of reading CSV
Taken from mock exam 
```python
session.read.format('csv').option('field_delimiter', ',').option('skip_header', 1).load('@my_data_stage')
```
This shows the correct, chained-method syntax for the Snowpark `DataFrameReader`. You first specify the file format (`.format('csv')`), then chain one or more `.option()` calls to configure how the file should be parsed (in this case, specifying the delimiter and indicating that the first row is a header to be skipped), and finally call `.load()` with the stage location to create the DataFrame.

<a id="From_Raw_SQL_Statements"></a>

### 2D. DataFrame from Raw SQL Statements

<a id="Session_SQL"></a>

#### 2Da. Creating `DataFrame` Objects Using `Session.sql(...)`

- Execute raw SQL statements
- Data represented could come from multiple sources
    - Joined tables and views
    - Data from system and user functions
    
```python
# Pass the query string to Session.sql(...)
results_sql_DF = (session
    .sql("<query string>") # Returns a DataFrame
)                
```

> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.Session.sql(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Session.sql.html#snowflake.snowpark.Session.sql)

In the example below, we join three tables, group on columns including a having clause, sort, and limit, all using typical Snowflake-standard SQL syntax. 

The result is a `DataFrame` that represents our result set - **not** a `Table`. 

In [13]:
# Create a query string
query_string = (
"""SELECT CUS.C_NAME AS CUSTOMER_NAME, SUB.OC AS ORDER_COUNT, SUB.CK AS CUSTOMER_ID, SUB.OK AS ORDER_ID, SUB.TQ::INT AS LINEITEM_QUANTITY_SUM
    FROM TRAINING_DB.TPCH_SF10.CUSTOMER CUS 
        JOIN (SELECT CU.C_CUSTKEY AS CK, ORD.O_ORDERKEY AS OK, SUM(LI.L_QUANTITY) AS TQ, COUNT(ORD.O_ORDERKEY) AS OC 
                FROM TRAINING_DB.TPCH_SF10.LINEITEM LI 
                    JOIN TRAINING_DB.TPCH_SF10.ORDERS ORD 
                        ON LI.L_ORDERKEY = ORD.O_ORDERKEY 
                    JOIN TRAINING_DB.TPCH_SF10.CUSTOMER CU 
                        ON CU.C_CUSTKEY = ORD.O_CUSTKEY              
              GROUP BY CU.C_CUSTKEY, ORD.O_ORDERKEY HAVING SUM(LI.L_QUANTITY) > 100) SUB 
           ON CUS.C_CUSTKEY = SUB.CK   
    ORDER BY LINEITEM_QUANTITY_SUM DESC, ORDER_COUNT DESC, CUSTOMER_ID ASC, ORDER_ID ASC
    LIMIT 12"""
)    

# Pass the query string to Session.sql(...)
results_sql_DF = (session
    .sql(query_string)
)                

# Print the class of the resulting object
print(f"The class of results_sql_DF is {type(results_sql_DF)}")

The class of results_sql_DF is <class 'snowflake.snowpark.dataframe.DataFrame'>


The schema will be retrieved when needed. One example of "when needed" would be to ask for the schema using `.schema`. 

In [14]:
results_sql_DF.schema

StructType([StructField('CUSTOMER_NAME', StringType(25), nullable=False), StructField('ORDER_COUNT', LongType(), nullable=False), StructField('CUSTOMER_ID', LongType(), nullable=False), StructField('ORDER_ID', LongType(), nullable=False), StructField('LINEITEM_QUANTITY_SUM', LongType(), nullable=True)])

***You won't see a call to retrieve the schema in the query history section of Snowsight.*** Under the hood, Snowpark is using a prepared statement to retrieve the schema of the table. This will be done the first time an operation on the `DataFrame` requires the schema such as a call to `DataFrame.schema()`, `DataFrame.with_column(...)`, `DataFrame.join(...)`, `DataFrame.col(...)` etc.

In [15]:
# View some data
results_sql_DF.show(5)

---------------------------------------------------------------------------------------------
|"CUSTOMER_NAME"     |"ORDER_COUNT"  |"CUSTOMER_ID"  |"ORDER_ID"  |"LINEITEM_QUANTITY_SUM"  |
---------------------------------------------------------------------------------------------
|Customer#000000384  |4              |384            |518         |195                      |
|Customer#000000265  |4              |265            |426         |184                      |
|Customer#000000107  |4              |107            |1946        |178                      |
|Customer#000000328  |4              |328            |222         |176                      |
|Customer#000000157  |4              |157            |1421        |175                      |
---------------------------------------------------------------------------------------------




---

<a id="DataFrame_Schemas_and_Data_Types"></a>

## 3. DataFrame Schemas and Data Types

Before looking at Dataframe transformations. Let's take a look at Data Types and Schemas

<a id="Snowpark_SQL_Datatypes"></a>
### 3A. Snowpark Data Types and Snowflake SQL Data Types

Snowpark data types reside in the package `snowflake.snowpark.types`. 

They range from omnipresent data types like integers and strings to non-standard data types like arrays and geospatial data. The base class for these Snowpark data types is `snowflake.snowpark.types.DataType()`.

<img src="../../images/SnowparkDataTypes.png" alt="Snowpark Types" width=65% /> 

When creating a `DataFrame` from a Snowflake table or view, Snowpark will perform the data conversion according to the table above.



> **&#128221; Note:** See documentation for further details:
> - [Snowpark Data Types](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/types.html#data-types)

<a id="Schemas"></a>
### 3B. Schemas

#### Schemas in DataFrames vs. Schemas in Snowflake Databases

- `DataFrame` schema = defined structure
    - Number and order of columns
    - Names and data types of columns
    - Needed for all `DataFrame`s when performing business logic
        - May be explicitly provided when created from a `list` or staged CSV file
        - May be inferred when created from Snowflake table, view, or from staged binary file format
    - Object of type `snowflake.snowpark.types.StructType`
    - Access by invoking `DataFrame.schema`
- Not to be confused with Snowflake database schema
    - Objects organized into namespaces
    - `<database Name>.<schema Name>.<table Name>`

#### Creating Schemas Using `StructType` and `StructField`

- Schema is an object of type `snowflake.snowpark.types.StructType`
- Each `StructType` consists of a sequence of `StructField` instances

```python
schema = (StructType(
    [
      StructField("<col name>", <some>Type),
      StructField("<col name>", <some>Type),
      StructField("<col name>", <some>Type),
      ...
     ]
  )
) 
```

> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.types.StructType](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.types.StructType.html#snowflake-snowpark-types-structtype)
> - [snowflake.snowpark.types.StructField](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.types.StructField.html#snowflake-snowpark-types-structfield)

---

We create the schema by passing a sequence of `StructField` instances to the constructor of a new `StructType`.

In [ ]:
from snowflake.snowpark.types import StructType,StructField,IntegerType,StringType

# Create a StructType of 3 StructField objects
friends_schema = (
    StructType([
         StructField("ID", IntegerType())    # Column 1
        ,StructField("NAME", StringType())   # Column 2
        ,StructField("STATUS", StringType()) # Column 3
    ]
  )
)

print(f"The schema class is: {type(friends_schema)}")

#### Accessing the Schema Using `.schema`

```python
mySchema = myDF.schema
```

In [ ]:
print("The create statement for the first DataFrame in this lesson")
print(("""df_list = (session
            .create_dataframe(
                ["some", "data","in","a list"]
            )
        )""" 
      )
)

# Implicitly prints the schema to standard out in a Jupyter notebook
print("The schema of that DataFrame")
df_list.schema

<a id="intro_dataframe_transformations"></a>
## 4. Introduction to DataFrame Transformations

<a id="DataFrame_TransformationsI"></a>

#### Transformations return a new DataFrame with its contents altered
- Some common transformations will:
    - Add, remove, rename or reorder columns
    - Alter contents of data values in columns
    - Filter rows based on certain column values
    - Order rows based on certain column values
- Additionally, some more complex transformations will:
    - Combine multiple data sets into one
    - Reshape nested data into a tabular format
    - Summarize data based on aggregate results (sum, min, max, etc.)
- `DataFrame` transformations are evaluated lazily
    - Transformations are noted but not executed until an action is invoked
    - Until an action is invoked, no transform operations are executed on the Snowflake Virtual Warehouse
- `DataFrame`s are immutable; each transformation returns a new `DataFrame` object
- `DataFrame` operations can be chained together to create succinct and readable code

<img src="../../images/DFOperationsTransformPython.png" alt="DFOperationsTransform" style="width:85%;display:block;margin-left:5%;" />


<a id="Accessing_Columns"></a>

### 4A. Accessing `Column` Objects

There are several different ways to access the columns in a `DataFrame`. See the appendix, [*Referencing Column Objects*](../Appendices/Appendix-Referencing-Column-Objects.ipynb). Look for the section headed *The Many Ways to Reference A Column Object*.

For this course, we will primarily use the function `functions.col(...)`.

```python
from snowflake.snowpark.functions import col
my_df.select(col("column_name"))
```

<div style="font-size: 0;height: 20px;line-height: 0;"></div>

We will need a `DataFrame` for demo purposes. Below we create a `DataFrame` representing a few columns from the `LINEITEM` table in a sample database. 

In [ ]:
from snowflake.snowpark.functions import col

line_items_df = (session.table("TRAINING_DB.TPCH_SF10.LINEITEM") # MyNote: TRAINING_DB is my location for this. 
    .select( 
         col("L_QUANTITY")
        ,col("L_EXTENDEDPRICE")
        ,col("L_DISCOUNT")
        ,col("L_TAX")
        ,col("L_SHIPINSTRUCT")
    )
)

print("The schema:")
print(type(line_items_df.schema))
for field in line_items_df.schema.fields:
    print(field)

The schema:
<class 'snowflake.snowpark.types.StructType'>
StructField('L_QUANTITY', DecimalType(12, 2), nullable=False)
StructField('L_EXTENDEDPRICE', DecimalType(15, 2), nullable=False)
StructField('L_DISCOUNT', DecimalType(15, 2), nullable=False)
StructField('L_TAX', DecimalType(15, 2), nullable=False)
StructField('L_SHIPINSTRUCT', StringType(25), nullable=False)


<a id="Using_the_com_snowflake_snowpark_functions_Object"></a>

### 4B. Using the `snowflake.snowpark.functions` Object   

The Snowpark APIs come with a built-in helper object named `functions`.

The vast majority of operations in the `functions` object return an object of type `Column`. We can then use these functions wherever a `Column` object is needed. 


1. Use the fully-qualified name of any function in the `functions` object.


```python
from snowflake.snowpark import functions
from snowflake.snowpark.functions import col

my_new_df = (my_df
 .select(
      col("<some column>")
     ,functions.lit("Hello World") # functions.functionName(...)
     ,functions.random()           # functions.functionName(...)
    )
 .yada(...)
)           
```


The `functions` object contains several functions that are aliases of other functions. This can make code migration easier on users. 

<br/>
<br/>

> **&#128221; Note:** See documentation for further details:
> - [Functions Home](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/functions.html#functions)
> - [snowflake.snowpark.functions.lit(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.functions.lit.html#snowflake.snowpark.functions.lit)
> - [snowflake.snowpark.functions.random()](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.functions.random.html#snowflake.snowpark.functions.random)
> - [GitHub `functions.py` aliases](https://github.com/snowflakedb/snowpark-python/blob/main/src/snowflake/snowpark/functions.py#L8187)

<a id="Adding_a_Column_Using_DataFrame_with_column_and_withColumns"></a>

### 4C. Adding a `Column` Using `DataFrame.with_column(...)` and `.with_columns(...)`

The function `DataFrame.with_column(...)` appends a new `Column` object to the `DataFrame`. 

Imagine you want to add a `Column` to your `DataFrame` that contains a literal value such as `"Hello"`, `3.14`, or `False`. This is easily achievable with the function `functions.lit(...)`. The name of the function is short for "literal," and it returns a `Column` object that can be passed to the function `DataFrame.with_column(...)`. The syntax would be as follows:

```python
from snowflake.snowpark.functions import lit
my_new_df = (my_df
    .withColumn("<col name>", lit(someLiteralValue))
    .with_column("<col name>", someColumnObject)
    .with_column("<col name>", someColumnObject)
)
```

**Note:** The function `with_column(...)` has an alias `withColumn(...)`. 

> **&#128221; Note:** See documentation for further details:
> - [Column Home](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/column.html#column)
> - [snowflake.snowpark.DataFrame.with_column(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.DataFrame.with_column.html#snowflake.snowpark.DataFrame.with_column)
> - [snowflake.snowpark.DataFrame.withColumn(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.DataFrame.withColumn.html#snowflake.snowpark.DataFrame.withColumn)

In [ ]:
# Import any operations from the functions object that we need
from snowflake.snowpark.functions import lit

# Create a DataFrame for demo purposes
# MyNote: The data types for the new columns are being set implicitly from the literal values being passed in.
new_df = (line_items_df
    .with_column("txt", lit("Hello World"))
    .with_column("num", lit(3.14))
    .with_column("bool", lit(False))
)           
for field in new_df.schema.fields:
    print(field)

# View some data
new_df.show(3)   

StructField('L_QUANTITY', DecimalType(12, 2), nullable=False)
StructField('L_EXTENDEDPRICE', DecimalType(15, 2), nullable=False)
StructField('L_DISCOUNT', DecimalType(15, 2), nullable=False)
StructField('L_TAX', DecimalType(15, 2), nullable=False)
StructField('L_SHIPINSTRUCT', StringType(25), nullable=False)
StructField('TXT', StringType(11), nullable=False)
StructField('NUM', DoubleType(), nullable=False)
StructField('BOOL', BooleanType(), nullable=True)
----------------------------------------------------------------------------------------------------------------
|"L_QUANTITY"  |"L_EXTENDEDPRICE"  |"L_DISCOUNT"  |"L_TAX"  |"L_SHIPINSTRUCT"   |"TXT"        |"NUM"  |"BOOL"  |
----------------------------------------------------------------------------------------------------------------
|34.00         |8436.15            |0.00          |0.04     |DELIVER IN PERSON  |Hello World  |3.14   |False   |
|37.00         |11575.76           |0.05          |0.06     |DELIVER IN PERSON  |Hello W

## MyNote Column data types using with_column()

CLINE info:

`with_column()` itself has no type parameter — its signature is simply:

```python
df.with_column(col_name: str, col: Column) -> DataFrame
```

The type is determined entirely by the `Column` expression you pass in. To control the type explicitly, you chain `.cast()` onto the column expression *before* passing it to `with_column()`:

---

### Pattern: `lit(...).cast(SomeType())`

```python
from snowflake.snowpark.functions import lit
from snowflake.snowpark.types import DecimalType, IntegerType, StringType

new_df = (line_items_df
    .with_column("txt",  lit("Hello World"))                      # StringType (implicit)
    .with_column("num",  lit(3.14).cast(DecimalType(10, 2)))      # DecimalType(10,2) explicit
    .with_column("flag", lit(1).cast(BooleanType()))              # BooleanType explicit
    .with_column("qty",  col("L_QUANTITY").cast(IntegerType()))    # cast existing column
)
```

---

### Pattern: cast an existing column expression

```python
from snowflake.snowpark.types import LongType

# The notebook does exactly this:
df.with_column("QUANTITY_INT", col("L_QUANTITY").cast(LongType()))
```

---

### Summary

| Goal | How |
|---|---|
| Implicit type from Python value | `lit(value)` — type inferred |
| Explicit type on a literal | `lit(value).cast(SomeType())` |
| Explicit type on an existing column | `col("name").cast(SomeType())` |
| Explicit type on any expression | `some_expression.cast(SomeType())` |

`.cast()` is available on any `Column` object, so it composes naturally with any expression — not just literals. The notebook uses this pattern in the casting section (section 4Fa) where `col("L_QUANTITY").cast(LongType())` is demonstrated.

## MyNote: New Column based on value of existing Column


### 1. Simple column reference / copy

```python
# Snowpark
df.with_column("QUANTITY_COPY", col("L_QUANTITY"))

# pandas
df["QUANTITY_COPY"] = df["L_QUANTITY"]
# or chainable:
df.assign(QUANTITY_COPY=df["L_QUANTITY"])
```

---

### 2. Arithmetic on an existing column

```python
# Snowpark
df.with_column("DISCOUNTED_PRICE", col("L_EXTENDEDPRICE") * (lit(1) - col("L_DISCOUNT")))

# pandas
df["DISCOUNTED_PRICE"] = df["L_EXTENDEDPRICE"] * (1 - df["L_DISCOUNT"])
# or chainable:
df.assign(DISCOUNTED_PRICE=lambda x: x["L_EXTENDEDPRICE"] * (1 - x["L_DISCOUNT"]))
```

---

### 3. Cast to a different type

```python
# Snowpark
df.with_column("QUANTITY_INT", col("L_QUANTITY").cast(LongType()))

# pandas
df["QUANTITY_INT"] = df["L_QUANTITY"].astype(int)
```

---

### 4. Conditional logic (`when/otherwise` → `np.where` or `pd.cut`)

```python
# Snowpark
df.with_column("DISCOUNT_BAND",
    when(col("L_DISCOUNT") == 0,          lit("NO DISCOUNT"))
    .when(col("L_DISCOUNT") <= lit(0.05), lit("LOW"))
    .when(col("L_DISCOUNT") <= lit(0.08), lit("MEDIUM"))
    .otherwise(                           lit("HIGH"))
)

# pandas — using numpy.select (multiple conditions)
import numpy as np
conditions = [
    df["L_DISCOUNT"] == 0,
    df["L_DISCOUNT"] <= 0.05,
    df["L_DISCOUNT"] <= 0.08,
]
choices = ["NO DISCOUNT", "LOW", "MEDIUM"]
df["DISCOUNT_BAND"] = np.select(conditions, choices, default="HIGH")

# pandas — simple two-way: np.where
df["IS_DISCOUNTED"] = np.where(df["L_DISCOUNT"] > 0, "YES", "NO")
```

---

### 5. String functions

```python
# Snowpark
df.with_column("SHIPINSTRUCT_UPPER", upper(col("L_SHIPINSTRUCT")))

# pandas
df["SHIPINSTRUCT_UPPER"] = df["L_SHIPINSTRUCT"].str.upper()
df["SHIP_ABBREV"]        = df["L_SHIPINSTRUCT"].str[:4]
```

---

### 6. Combining two columns

```python
# Snowpark
df.with_column("TOTAL_VALUE",
    col("L_EXTENDEDPRICE") * (lit(1) - col("L_DISCOUNT")) * (lit(1) + col("L_TAX"))
)

# pandas
df["TOTAL_VALUE"] = df["L_EXTENDEDPRICE"] * (1 - df["L_DISCOUNT"]) * (1 + df["L_TAX"])
```

---

### Quick reference table

| Snowpark | pandas |
|---|---|
| `col("x")` | `df["x"]` |
| `lit(value)` | just use the value directly |
| `.cast(LongType())` | `.astype(int)` |
| `when(...).otherwise(...)` | `np.select(...)` / `np.where(...)` |
| `upper(col("x"))` | `df["x"].str.upper()` |
| `substring(col("x"), lit(1), lit(4))` | `df["x"].str[:4]` |

The main pattern difference: in pandas you reference columns as `df["col_name"]` directly in expressions, whereas in Snowpark you use `col("col_name")` to get a `Column` object that can be composed into expressions.

---

#### The general rule

The second argument to `with_column()` is any `Column` expression. `col("name")` gives you a reference to an existing column, and you can apply any operator or function to it — arithmetic, string, date, conditional, casting — and the result becomes the new column's value.


### 4D.`Column` Mathematical Operators (`+`,`-`,`*`,`/` and `%`)

`Column` objects have many functions built into them. Most return a new `Column` reflecting some change to a source `Column` instance or instances. The API for `Column` contains overloaded operators such as `+` (add), `-` (subtract), `*` (multiply), etc. These operations act upon the field data that the objects represent. These operators produce new `Column` objects.


In [ ]:
# Import the col and lit functions
from snowflake.snowpark.functions import col,lit

# Create an in-memory DataFrame for demo purposes and transform it
(session.create_dataframe([""])
    .select(
         lit(10).alias("X")
        ,lit(3).alias("Y")
        ,(col("X") / col("Y")).alias("DIV")
        ,(col("X") * col("Y")).alias("MULT")
        ,(col("X") + col("Y")).alias("ADD")
        ,(col("X") - col("Y")).alias("SUB")
        ,(col("X") % col("Y")).alias("MOD (a.k.a. remainder)")
      )
    .show(5)
) 

<a id="Logical_Relational_and_Comparison_Operators"></a>
### 4E. Logical, Relational and Comparison Operators (`<`, `>`, `!=`, `&` etc.)

The `Column` API also contains logical `&` (and) and `|` (or), and comparison operators `<` (less than), `>` (greater than), etc. Comparisons to Python's `None` produce a SQL `NULL`. 



In [ ]:
print("Use > (greater than) Column operator")
(session.create_dataframe(["lit(10) > lit(3)"])
    .with_column_renamed(col("_1"), "Operator > (greater than)")
    .with_column("result",lit(10) > lit(3))
    .show()
)

<a id="Column_Expression_Functions"></a>

### 4F. Casting and Aliasing `Column` Objects

<a id="Casting_a_Column"></a>

#### 4Fa. Casting a `Column`

Casting is a commonly used feature in Snowpark. By printing the schema of our line item `DataFrame` below, we see that the "L_QUANTITY" column is of type `DecimalType(12,2)`. Below that, we cast the quantity column from a `DecimalType(12,2)` to a `LongType`. 

In [ ]:
# Import any operations from the functions object that we need
from snowflake.snowpark.types import LongType

print("Notice L_QUANTITY is of type DecimalType(12,2)")
for field in line_items_df.schema.fields:
    print(field)

# Create a DataFrame and extract the schema
schema = (line_items_df.select(
             col("L_QUANTITY").cast(LongType())
            ,col("L_EXTENDEDPRICE")
        ).schema
)

print("Looping through the StructField objects in a DataFrame's schema")
for field in schema.fields:
    print(field)

<a id="Aliasing_a_Column_Name"></a>

#### 4Eb. Aliasing a `Column` Name

A name like `CAST (""L_QUANTITY"" AS BIGINT)` is not very user-friendly. It can make referencing the column in later steps very difficult and brittle. We can rename a column with `.name(...)`, `.as_(...)`, or `.alias(...)`. The class `DataFrame` contains a `rename(...)` function as well.

> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.Column.name(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Column.name.html#snowflake.snowpark.Column.name)
> - [snowflake.snowpark.Column.as_(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Column.as_.html#snowflake.snowpark.Column.as_)
> - [snowflake.snowpark.Column.alias(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Column.alias.html#snowflake.snowpark.Column.alias)

In [ ]:
# Import any type objects we need for casting
from snowflake.snowpark.types import LongType

print("Using 4 ways to rename a Column")
(line_items_df
    .select(
         col("L_QUANTITY").cast(LongType()).name("QUANTITY")
        ,col("L_EXTENDEDPRICE").alias("EXTENDED_PRICE")
        ,col("L_TAX").as_("TAX")
        ,col("L_SHIPINSTRUCT")
      )
    .rename(col("L_SHIPINSTRUCT"), "SHIPPING_INSTRUCTIONS")
    .show()
) 

---
<a id="Aggregating_Data_Using_DataFrame_group_by_agg"></a>

## 4G. Aggregating Data Using `DataFrame.group_by(...).agg(...)`

Often, aggregations are desired on groups of rows rather than all rows. We can group our data first and *then* perform aggregations on the groupings. We achieve this through `DataFrame.group_(...).agg(...)`. The arguments to `.group_by(...)` can be a single string, a single `Column` object, or a `List` containing strings or `Column` objects.  In Snowpark for Python `v0.12.0` Snowflake enabled `RelationalGroupedDataFrame.agg(...)` to take a variable number of arguments of `Column`s, `tuple`s, or `list`s.

**Note:** `DataFrame.groupBy(...)` is an alias for `DataFrame.group_by(...)`.

```python
from snowflake.snowpark.functions import col, sum, max, min, count

(my_df.group_by(col("<grouping column>"))
     .agg(
         sum(col("<numeric column>"))
        ,min(col("<numeric column>"))
        ,max(col("<numeric column>"))
        ,count("*")
     )
     .show()
)
```

> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.DataFrame.group_by(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.DataFrame.group_by.html#snowflake-snowpark-dataframe-group-by)
> - [snowflake.snowpark.RelationalGroupedDataFrame](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.RelationalGroupedDataFrame.html#snowflake-snowpark-relationalgroupeddataframe)
> - [snowflake.snowpark.RelationalGroupedDataFrame.agg(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.RelationalGroupedDataFrame.agg.html#snowflake-snowpark-relationalgroupeddataframe-agg)


In [ ]:
# Import any operations from the functions object we need
from snowflake.snowpark.functions import col, avg, median, count

(line_items_df
     .group_by(col("L_SHIPINSTRUCT")) # Produces a RelationalGroupedDataFrame
     .agg(                            # Produces a DataFrame
        [ count("*").alias("Item Count")
         ,avg(col("L_DISCOUNT")).alias("Average Discount")
         ,median(col("L_EXTENDEDPRICE")).alias("Median Price")
        ]
      ) # End agg(...)
     .show(4)
)

---
<a id="DataFrame_Actions"></a>
## 5. DataFrame Actions

#### Create and transform operations do not materialize data
- Executed lazily; no data is materialized until an action is invoked
- Like a recipe, the steps are noted

#### Actions materialize data
- Executed eagerly
- Once an action is invoked, all previously noted create and transform operations for the `DataFrame` are evaluated
- Statements are executed on the server and can be seen in query history
- Actions let you see, gather, or persist `DataFrame` data
- Like gathering ingredients and executing the steps of a recipe and then... you get a cake!

<img src="../../images/DFOperationsActionPython.png" alt="DFOperationsAction" style="width:85%;display:block;margin-left:5%;" />

---
<a id="DataFrame_collect"></a>

## 5A.  DataFrame Collect

A very common action used with `DataFrame` objects is `DataFrame.collect()`. This action will cause the SQL for the `DataFrame` to be generated and sent to your Snowflake account similarly to `DataFrame.show()`. However, the results will **not** print to standard out in the same way `.show()` does. The return of `DataFrame.collect()` is a `list` of `Row` objects. 


**Note:** Using the actions `DataFrame.collect()` and `DataFrame.to_local_iterator()` are covered in more depth in the appendex titled, [*Working with Row Data*](../Appendices/Appendix-Working-with-Row-Data.ipynb).

In [ ]:
rows = session.table("TRAINING_DB.TPCH_SF10.CUSTOMER").limit(5).collect()

for row in rows:
    print(row)
print("---")    
for row in rows:
    print(f"ID:{row['C_CUSTKEY']}\t Phone:{row.C_PHONE}")

### MyNote re DataFrame Collect
From Exam:
A data scientist creates a Snowpark DataFrame from a table that has 1 billion rows. The data scientist then calls the `.collect()` method on this DataFrame without any prior filtering or aggregation. What is the most likely outcome?

The client machine running the Python script will run out of memory, and the script will crash with an `OutOfMemoryError`.

The `.collect()` method is an action that attempts to bring the entire result set of a query into the memory of the client machine. For a 1-billion-row table, this will require a massive amount of memory (likely hundreds of gigabytes or more) that a typical client machine does not have, leading to a fatal `OutOfMemoryError`.

It is absolutely critical for a Snowpark developer to understand that actions like `.collect()` and `.to_pandas()` move data from the Snowflake warehouse to the client machine's memory. These methods should never be used on a raw, large DataFrame. They should only be called after the data has been significantly reduced in size through filtering, aggregation, or sampling operations. Failure to do so is one of the most common causes of application failures when using Snowpark.

---
<a id="Creating_Views_from_DataFrames"></a>

## 5B.  Creating Views from DataFrames

Snowflake views can be created from `DataFrame` objects. 


```python
my_df.create_or_replace_view(...)
my_df.create_or_replace_temp_view(...)

```


**Note:** Both the create view methods have aliases.
-  `createOrReplaceView(...)` is an alias for `create_or_replace_view(...)`
-  `createOrReplaceTempView(...)` is an alias for `create_or_replace_temp_view(...)`


In [ ]:
# Import needed functions
from snowflake.snowpark.functions import col, avg

view_name = "AVG_PRICE_BY_SHIPPING"

# Create a DataFrame of aggregations over the line item data
ship_avg_df = (session.table("TRAINING_DB.TPCH_SF10.LINEITEM")
    .group_by(col("L_SHIPINSTRUCT"))
    .agg(avg(col("L_EXTENDEDPRICE")).alias("AVERAGE_EXTENDED_PRICE"))
    .rename(             
         col("L_SHIPINSTRUCT")
        ,"SHIPPING_INSTRUCTIONS"
      )
) 

# Create the first view
ship_avg_df.create_or_replace_view(view_name)

print(f"View {view_name} created") 

---
<a id="Writing_a_DataFrame_to_a_Table"></a>

## 5C. Writing a `DataFrame` to a Table

There will come a time when you want to take the state of the data your `DataFrame` represents and save it to a table in your Snowflake account. The table might already exist, but this is not a requirement. The first step is to obtain a `DataFrameWriter` from your `DataFrame`. 

We can obtain a `DataFrameWriter` by accessing the property `write` on a `DataFrame`.

```python
my_dataframe_writer = my_df.write
```

Below we join several Dataframes and write them to a table

In [ ]:
# Imports we will need
from snowflake.snowpark.functions import col, sum, count
from snowflake.snowpark.types import IntegerType
from copy import copy

# Grab some customer info, rename the columns for clarity
customer_df = (
    session.table("TRAINING_DB.TPCH_SF10.CUSTOMER")
        .select(
             col("C_NAME").alias("CUSTOMER_NAME")
            ,col("C_CUSTKEY").alias("CUSTOMER_ID")
     )
)

print("customer_df created:")
(customer_df
    .show(3)
) 

# Grab some orders info, rename the columns for clarity
orders_df = (
    session.table("TRAINING_DB.TPCH_SF10.ORDERS")
        .select(
             col("O_ORDERKEY").alias("ORDER_ID")
            ,col("O_CUSTKEY").alias("ORDER_CUSTOMER_ID")
     )
)    

print("orders_df created:")
(orders_df
    .show(3)
)

# Grab some line item data, rename the columns for clarity
line_items_df = (
    session.table("TRAINING_DB.TPCH_SF10.LINEITEM")
        .select(
             col("L_ORDERKEY").alias("LINEITEM_ORDER_ID")
            ,col("L_QUANTITY").alias("LINEITEM_QUANTITY")
     )
) 

print("lineitems_df created:")
(line_items_df
    .show(3)
) 

# Join, group, aggregate, filter, sort, and limit
results_df = (
    orders_df
        .join(customer_df, col("CUSTOMER_ID") == col("ORDER_CUSTOMER_ID"))
        .join(line_items_df, col("LINEITEM_ORDER_ID") == col("ORDER_ID"))
        .group_by(col("CUSTOMER_ID"), col("ORDER_ID"))
        .agg([
                 sum(col("LINEITEM_QUANTITY")).cast(IntegerType()).alias("LINEITEM_QUANTITY_SUM") 
                ,count(col("LINEITEM_ORDER_ID")).alias("ORDER_LINE_COUNT")
             ]
         )
        .join(copy(customer_df), "CUSTOMER_ID")
        .where(col("LINEITEM_QUANTITY_SUM") > 100)
        .select(
                 col("CUSTOMER_NAME")
                ,col("ORDER_ID")
                ,col("ORDER_LINE_COUNT")
                ,col("CUSTOMER_ID")
                ,col("LINEITEM_QUANTITY_SUM")
             )
         .sort(  
                 col("LINEITEM_QUANTITY_SUM").desc()
                ,col("ORDER_LINE_COUNT").desc()
                ,col("CUSTOMER_ID").asc()
                ,col("ORDER_ID").asc()
             )
        .limit(10)  
)

print("results_df created:")
(results_df
    .show()
) 

# Access the writer object
print("Accessing the DataFrameWriter of our DataFrame")
results_dfw = results_df.write

# Class of results_dfw
print("The class of results_dfw is:")
print(type(results_dfw))

In [ ]:
# Come up with a name for our table
desired_table_name = "TOP_10_LINE_ITEM_QUANTITY_CUSTOMERS"
  

# Configure the DataFrameWriter and use it to save to the desired table
print("Saving the table using Append as the mode")
(results_df
    .write
    .mode("Append")                    # Configure mode 
    .save_as_table(desired_table_name) # Write to storage layer
)

---

<a id="Explain_Plans"></a>
## 6. Explain And Describe

<a id="DataFrame_explain"></a>
### 6A. Using `DataFrame.explain()`

We can look at the explain plan using `DataFrame.explain()`. 

> **&#128221; Note:** The function `.explain()` is not an action. No query is executed on your Snowflake account when invoking  `.explain()`.

> **&#128221; Note:** Prints the query execution plan if only a single SELECT/DML/DDL statement will be executed.
> - [snowflake.snowpark.DataFrame.explain()](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.DataFrame.explain.html#snowflake.snowpark.DataFrame.explain)

In [ ]:
results_sql_DF.explain()

<a id="DataFrame_describe"></a>
### 6B. Using `DataFrame.describe()`

Using `DataFrame.describe()` can compute basic statistics for numeric columns, which includes `count`, `mean`, `stddev`, `min`, and `max`.


> **&#128221; Note:** The function `.describe()` is a transformation. The lineage of the `DataFrame` will be converted to SQL and be embedded with statistic-gathering operations to produce new `DataFrame`.  An action is required to cause the SQL to be sent to our Snowflake account and return results. 

For details on using `DataFrame.describe()`, see the [API reference documentation on `describe()`.](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/api/snowflake.snowpark.DataFrame.describe)
<br />


In [ ]:
(results_sql_DF
     .describe() # Transformation! Produces a new DataFrame
     .show()     # Action! Causes the stat-gathering SQL to be sent to our Snowflake account
)

---
<a id="L2_cleanup"></a>

## 7. Clean Up and Close Session

Best practice is to clean up demo objects, suspend our warehouse, and close the Snowpark Session object.

In [ ]:
close_session_and_clean_up(get_lesson())

### &#10071; `Shut Down Kernel`
> After completing the activities in a notebook and before moving on to the next exercise, shut down the completed notebook by right-clicking on the notebook name and selecting `Shut Down Kernel`.